# 04 — Pixel explorer

Shows the time series (VV, VH, VH − VV, rain, platforms) of a single 10 m pixel, reading only that pixel from the stack. How to read the curves: `docs/01_sar_basics.md` §5.

> **Safe to re-run:** finished work is skipped. If the kernel dies or a cell crashes, just run the same cells again.

## Setup

**Which config is used?** After a run is chosen, everything uses the run's **frozen** config (`runs/<run_id>/run_config.yaml`), so this run's files are always read with the settings that produced them. If you edited your own config since `new-run`, a warning lists the differing keys; such changes need a **new run**. `auth.project` also stays the run's project. Taken from your config instead: `qa.acknowledged_issues` (accepting QA issues is a decision made after the run), `auth.key_file` and `resources` (they describe the machine, so a run exported on a laptop can be continued on a cloud notebook server).

In [ ]:
from pathlib import Path
import sys

# Project root = parent of notebooks/. Adding src/ is only needed if you did not run `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# >>> Change this to YOUR config file (copied from config/pipeline.example.yaml; it must live in config/) <<<
CONFIG_PATH = ROOT / "config" / "my_aoi_season.yaml"

from sar_pipeline import config, resources
user_cfg = config.load_config(CONFIG_PATH)   # your editable config; the RUN config is loaded below
print("Config :", CONFIG_PATH)
print("Machine:", resources.detect_resources(user_cfg).describe())

from sar_pipeline.cli import load_run_context

RUN_ID = None   # None = newest run; or a run folder name such as "v001_20260915"
cfg, run_path = load_run_context(user_cfg, RUN_ID)   # frozen run config from here on
print("Run:", run_path)
TRACK = cfg["s1"]["tracks"][0]["track_id"]   # or any selected track id
print("Track:", TRACK)

## Pick a location

Set `LON`/`LAT` to a point inside your AOI (e.g. a known field). If left as `None`, a point inside the AOI polygon is chosen automatically. Do not save the notebook with coordinates or outputs: clear outputs before committing.

In [ ]:
import geopandas as gpd

LON, LAT = None, None   # <- e.g. copy from QGIS

if LON is None or LAT is None:
    aoi = gpd.read_file(config.aoi_path(cfg)).to_crs("EPSG:4326")
    p = aoi.union_all().representative_point()
    LON, LAT = p.x, p.y
print(f"lon={LON:.6f} lat={LAT:.6f}")

## Time series by lon/lat

Columns: `<POL>_db` per polarisation, `VH_minus_VV_db` when both exist, rain, platforms and `valid`.

In [ ]:
from sar_pipeline import pixel_query

df = pixel_query.pixel_timeseries_lonlat(cfg, run_path, TRACK, LON, LAT)
df

In [ ]:
ax = pixel_query.plot_timeseries(df, cfg=cfg, title=f"{TRACK} (lon/lat query)")

## Time series by pixel id

Every pixel has a stable id (`pid`). Convert between pid and coordinates with `sar_pipeline.index`.

In [ ]:
from sar_pipeline import grid, index

gd = grid.load_grid(cfg)
PID = index.lonlat_to_pid(gd, LON, LAT)
print("pid:", PID, "| row/col:", index.pid_to_rowcol(gd, PID), "| centre lon/lat:", index.pid_to_lonlat(gd, PID))

df_pid = pixel_query.pixel_timeseries(cfg, run_path, TRACK, PID)
ax = pixel_query.plot_timeseries(df_pid, cfg=cfg, title=f"{TRACK}  pid {PID}")

## Reading the plot

- **VH** rising over weeks then falling → a crop cycle; the length separates maize (~4 months) from sugarcane (~10–12 months).
- **Very low VV and VH** early in the season → flooded field (rice).
- A **single-date spike** with rain ≥ 5 mm in 24 h → weather, not growth.
- The first and last dates are noisier (fewer neighbours for the speckle filter; see `n_temporal_neighbors` in `dates.csv`).
- Vertical markers = Sentinel-1 constellation events (possible sensor step); shaded = long data gaps.